# Experiment No. 05 - Variational Autoencoder on MNIST

Run cells from top to bottom.

In [ ]:
!pip -q install tensorflow matplotlib

## Aim
Train a VAE on MNIST and generate new digit-like images.

In [ ]:
import tensorflow as tf, numpy as np, matplotlib.pyplot as plt
(x,_),_=tf.keras.datasets.mnist.load_data()
x=x.astype("float32").reshape(-1,784)/255
ds=tf.data.Dataset.from_tensor_slices(x).shuffle(10000).batch(128)
latent=2
enc=tf.keras.Sequential([tf.keras.layers.Input((784,)),tf.keras.layers.Dense(256,activation="relu"),tf.keras.layers.Dense(4)])
dec=tf.keras.Sequential([tf.keras.layers.Input((latent,)),tf.keras.layers.Dense(256,activation="relu"),tf.keras.layers.Dense(784,activation="sigmoid")])
opt=tf.keras.optimizers.Adam()
for epoch in range(5):
    for batch in ds:
        with tf.GradientTape() as tape:
            mu,lv=tf.split(enc(batch),2,axis=1)
            z=mu+tf.exp(0.5*lv)*tf.random.normal(tf.shape(mu))
            out=dec(z)
            rec=tf.reduce_sum(tf.keras.losses.binary_crossentropy(batch,out),axis=1)
            kl=-0.5*tf.reduce_sum(1+lv-mu**2-tf.exp(lv),axis=1)
            loss=tf.reduce_mean(rec+kl)
        vars=enc.trainable_variables+dec.trainable_variables
        opt.apply_gradients(zip(tape.gradient(loss,vars),vars))
    print("Epoch",epoch+1,"Loss",float(loss))
imgs=dec(tf.random.normal([10,2])).numpy()
plt.figure(figsize=(10,3))
for i in range(10):
    plt.subplot(2,5,i+1); plt.imshow(imgs[i].reshape(28,28),cmap="gray"); plt.axis("off")
plt.show()

## Result
The VAE learned a latent representation and generated MNIST-like digits.